In [1]:
import numpy as np
#I used the functions/library of functions from Gezerlis

In [8]:
A=np.array([4.,4,8,4,
            4,5,3,7,
            8,3,9,9,
            4,7,9,5]).reshape(4,4)
bs=np.array([1.,2,3,4])

In [15]:
#forsub
def forsub(L,bs):
  n=bs.size
  xs=np.zeros(n)
  for i in range(n):
    xs[i]=(bs[i]-L[i,:i]@xs[:i]/L[i,i])
  print(f"L:\n{L}\nb:\n{bs}\n")
  print(f"computed y (solution of Ly=b):\n{xs}\n")
  return xs

#backsub
def backsub(U,bs):
  n=bs.size
  xs=np.zeros(n)
  for k,i in enumerate(reversed(range(n))):
    xs[i]=(bs[i]-U[i,i+1:])@xs[i+1:]/U[i,i]
  print(f"U:\n{U}\ny:\n{bs}\n")
  print(f"computed x (solution of Ux=y):\n{xs}\n")
  return xs

We first attempt to solve the system of linear equations without pivoting. Chapter 4 from Gezerlis defines two functions that in this code I called: *LU_decomp* and *LU_solving*.

In [16]:
def LU_decomp(A):
  n=A.shape[0]
  L=np.identity(n)
  U=np.copy(A)

  for j in range(n-1):
    for i in range(j+1,n):
      coeff=U[i,j]/U[j,j]
      L[i,j]=coeff
      U[i,j:]-=coeff*U[j,j:]
  return L,U

def LU_solving(A,bs):
  L,U=LU_decomp(A)
  ys=forsub(L,bs)
  xs=backsub(U,ys)
  return xs

x_unknown=LU_solving(A,bs)
print(f"(without pivoting) x_unknown:\n{x_unknown}\n")

L:
[[ 1.   0.   0.   0. ]
 [ 1.   1.   0.   0. ]
 [ 2.  -5.   1.   0. ]
 [ 1.   3.  -0.5  1. ]]
b:
[1. 2. 3. 4.]

computed y (solution of Ly=b):
[1. 1. 6. 3.]

U:
[[  4.   4.   8.   4.]
 [  0.   1.  -5.   3.]
 [  0.   0. -32.  16.]
 [  0.   0.   0.   0.]]
y:
[1. 1. 6. 3.]

computed x (solution of Ux=y):
[nan nan nan nan]

(without pivoting) x_unknown:
[nan nan nan nan]



/tmp/ipython-input-3919151458.py:16: RuntimeWarning: invalid value encountered in scalar divide
  xs[i]=(bs[i]-U[i,i+1:])@xs[i+1:]/U[i,i]


There is no solution to the system of linear equations. Looking at U, we see the bottom row is just zeros. This shows that our matrix is rank deficient or has 0 determinant. Without pivoting, we are forced to divide by zeros which gives an undefined answer. To check, let's look at the rank and determinant of A:

In [17]:
#checking
print(f"determinant:\n{np.linalg.det(A)}\n")
print(f"rank:\n{np.linalg.matrix_rank(A)}\n")

determinant:
0.0

rank:
3



As shown, the determinant of A is 0 which means that it is singular and non-invertible. The rank is 3 instead of 4 which means that there are only 3 linearly independent equations is our system. Now, let's check if partial pivoting will do us any help. However, we already know that our system is rank-deficient and would provide no unique solutions.

In [19]:
def pivot(inA,inbs):
  A=np.copy(inA)
  bs=np.copy(inbs)
  n=bs.size

  for j in range(n-1):
    k=np.argmax(np.abs(A[j:,j]))+j
    if k!=j:
      A[[j,k]]=A[[k,j]]
      bs[j],bs[k]=bs[k],bs[j]

    denominator=A[j,j]
    if abs(denominator)<1e-19:
      print(f"Zero pivot at step{k} even affter pivoting. Matrix is singular")
      break

    for i in range(j+1,n):
      coeff=A[i,j]/A[j,j]
      A[i,j:]-=coeff*A[j,j:]
      bs[i]-=coeff*bs[j]

  if abs(A[-1,-1])<1e-19: #just checking the last step
    print("Cannot back-substitute: last pivot is zero (matrix is singular)")

  xs=backsub(A,bs)
  return xs

x_unknown=pivot(A,bs)
print(f"(with pivoting) x_unknown:\n{x_unknown}\n")

Cannot back-substitute: last pivot is zero (matrix is singular)
U:
[[ 8.          3.          9.          9.        ]
 [ 0.          5.5         4.5         0.5       ]
 [ 0.          0.         -4.36363636  2.18181818]
 [ 0.          0.          0.          0.        ]]
y:
[ 3.          2.5        -1.09090909 -2.        ]

computed x (solution of Ux=y):
[nan nan nan nan]

(with pivoting) x_unknown:
[nan nan nan nan]



/tmp/ipython-input-3919151458.py:16: RuntimeWarning: invalid value encountered in scalar divide
  xs[i]=(bs[i]-U[i,i+1:])@xs[i+1:]/U[i,i]


As we can see, pivoting does us no help. Which makes some sense because pivoting doesn't really change our rank-deficient matrix. In the end, our matrix is still singular and no unique solution comes out of it. Even if we replace the denominator with a small number to somehow force a solution out of it.

According to Numpy's module structure for numpy.linalg.solve, it is noted that,

> A must be square and of full-rank, i.e., all rows (or, equivalently, columns) must be linearly independent; if either is not true, use *lstsq* for the least-squares best "solution" of the system/equation

let's try their suggestion:


In [22]:
print(np.linalg.lstsq(A,bs))

(array([0.01, 0.27, 0.08, 0.11]), array([], dtype=float64), np.int32(3), array([2.40000000e+01, 5.00000000e+00, 4.00000000e+00, 4.71916469e-17]))


Somehow, a solution is brought up. However, note that Numpy's module structure clearly states that the function merely gives the best "solution" of the system. It is still not unique, but merely one of infinitely many.